<a href="https://colab.research.google.com/github/sankalp120/machinelearning/blob/main/mllab9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1: Import Libraries
This cell imports all necessary libraries for data manipulation, machine learning (scikit-learn, TensorFlow/Keras), and visualization.

In [8]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

from sklearn.datasets import load_iris

## Step 2: Load Data
This cell defines column names and loads the Iris dataset from a CSV file into a Pandas DataFrame. It also removes any empty rows.

In [9]:
# Load dataset using sklearn
iris = load_iris()
data = pd.DataFrame(data=np.c_[iris['data'], iris['target']],
                    columns=iris['feature_names'] + ['species'])

# Map numerical species to names to match original code's expectation for LabelEncoder
data['species'] = data['species'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Display the first few rows of the loaded data
display(data.head())

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## Step 3: Preprocess Data
This section prepares the data for the CNN model. It separates features (X) and target (y), encodes the species labels, converts them to a one-hot categorical format, scales the features, and reshapes them for the 1D CNN input.

In [10]:
# Features and target
X = data.iloc[:, 0:4].values
y = data.iloc[:, 4].values

# Encode species labels
encoder = LabelEncoder()
y = encoder.fit_transform(y)

# Convert to categorical (for CNN output layer)
y = to_categorical(y)

# Feature scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshape for CNN (samples, steps, channels)
X = X.reshape(X.shape[0], X.shape[1], 1)

print("Shape of preprocessed features (X):", X.shape)
print("Shape of preprocessed target (y):", y.shape)

Shape of preprocessed features (X): (150, 4, 1)
Shape of preprocessed target (y): (150, 3)


## Step 4: Split Data
This cell splits the preprocessed data into training and testing sets, allocating 80% for training and 20% for testing.

In [11]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (120, 4, 1)
X_test shape: (30, 4, 1)
y_train shape: (120, 3)
y_test shape: (30, 3)


## Step 5: Build and Compile CNN Model
This cell defines the architecture of a 1D Convolutional Neural Network (CNN) using Keras and then compiles the model with an Adam optimizer, categorical crossentropy loss, and accuracy as the evaluation metric.

In [12]:
# CNN Model
model = Sequential()

model.add(Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(MaxPooling1D(pool_size=2))

model.add(Flatten())

model.add(Dense(16, activation='relu'))
model.add(Dense(3, activation='softmax')) # 3 classes for Iris species

# Compile model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 3, 32)          │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 675 (2.64 KB)

 Trainable params: 675 (2.64 KB)

 Non-trainable params: 0 (0.00 B)

## Step 6: Train Model
This cell trains the compiled CNN model using the training data for a specified number of epochs and batch size.

In [13]:
# Train model
history = model.fit(X_train, y_train, epochs=50, batch_size=5, verbose=1, validation_split=0.1)

# Optionally, display training history
# import matplotlib.pyplot as plt
# plt.plot(history.history['accuracy'], label='accuracy')
# plt.plot(history.history['val_accuracy'], label='val_accuracy')
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy')
# plt.legend()
# plt.show()

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.3981 - loss: 1.1155 - val_accuracy: 0.5833 - val_loss: 1.0538
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3889 - loss: 1.0529 - val_accuracy: 0.5000 - val_loss: 1.0550
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5556 - loss: 1.0140 - val_accuracy: 0.5000 - val_loss: 1.0502
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5926 - loss: 0.9801 - val_accuracy: 0.5833 - val_loss: 1.0455
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6389 - loss: 0.9527 - val_accuracy: 0.5000 - val_loss: 1.0418
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6296 - loss: 0.9273 - val_accuracy: 0.5833 - val_loss: 1.0133
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6389 - loss: 0.8990 - val_accuracy: 0.5833 - val_loss: 1.0178
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6481 - loss: 0.8720 - val_accuracy: 0.5833 - val_lo

## Step 7: Evaluate Model
This cell makes predictions on the test set, converts the predictions and true labels back to class indices, and then evaluates the model's performance using accuracy, a confusion matrix, and a classification report.

In [14]:
# Prediction
y_pred = model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

# Evaluation
print("Accuracy:", accuracy_score(y_test_classes, y_pred_classes))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_classes, y_pred_classes))

print("\nClassification Report:")
print(classification_report(y_test_classes, y_pred_classes))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
Accuracy: 0.8333333333333334

Confusion Matrix:
[[8 2 0]
 [1 8 0]
 [2 0 9]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.80      0.76        10
           1       0.80      0.89      0.84         9
           2       1.00      0.82      0.90        11

    accuracy                           0.83        30
   macro avg       0.84      0.84      0.83        30
weighted avg       0.85      0.83      0.84        30

